In [1]:
import geopandas as gpd
import pandas as pd

df = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun.gpkg"
)

print(len(df))

df1 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-0.gpkg"
)

print(len(df1))

df2 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-1.gpkg"
)

print(len(df2))

561338
27042
13566


In [2]:
# Convert gml_id columns to sets
s0 = set(df["gml_id"])
s1 = set(df1["gml_id"])
s2 = set(df2["gml_id"])

# Pairwise overlaps
overlap_0_1 = s0 & s1
overlap_0_2 = s0 & s2
overlap_1_2 = s1 & s2

# Triple overlap
overlap_all = s0 & s1 & s2

print("========== DATASET SIZES ==========")
print(f"df  : {len(s0)} unique gml_id")
print(f"df1 : {len(s1)} unique gml_id")
print(f"df2 : {len(s2)} unique gml_id")

print("\n========== PAIRWISE OVERLAPS ==========")
print(f"df  ∩ df1 : {len(overlap_0_1)}")
print(f"df  ∩ df2 : {len(overlap_0_2)}")
print(f"df1 ∩ df2 : {len(overlap_1_2)}")

print("\n========== OVERLAP IN ALL 3 ==========")
print(f"df ∩ df1 ∩ df2 : {len(overlap_all)}")

print("\n========== UNIQUE (NO OVERLAP) ==========")

print(f"Only in df  : {len(s0 - s1 - s2)}")
print(f"Only in df1 : {len(s1 - s0 - s2)}")
print(f"Only in df2 : {len(s2 - s0 - s1)}")

========== DATASET SIZES ==========
df  : 561338 unique gml_id
df1 : 27042 unique gml_id
df2 : 13566 unique gml_id

========== PAIRWISE OVERLAPS ==========
df  ∩ df1 : 27042
df  ∩ df2 : 0
df1 ∩ df2 : 0

========== OVERLAP IN ALL 3 ==========
df ∩ df1 ∩ df2 : 0

========== UNIQUE (NO OVERLAP) ==========
Only in df  : 534296
Only in df1 : 0
Only in df2 : 13566


In [3]:
# Load parquet correctly
df3 = pd.read_parquet(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\predictions_checkpoint(error-rows)-0.parquet"
)

# Convert gml_id columns to sets
s1 = set(df1["gml_id"])
s3 = set(df3["gml_id"])

# Overlaps
overlap = s1 & s3

print("========== DATASET SIZES ==========")
print(f"df1 : {len(s1)} unique gml_id")
print(f"df3 : {len(s3)} unique gml_id")

print("\n========== OVERLAP ==========")
print(f"df1 ∩ df3 : {len(overlap)}")

print("\n========== UNIQUE ==========")
print(f"Only in df1 : {len(s1 - s3)}")
print(f"Only in df3 : {len(s3 - s1)}")

========== DATASET SIZES ==========
df1 : 27042 unique gml_id
df3 : 34430 unique gml_id

========== OVERLAP ==========
df1 ∩ df3 : 27042

========== UNIQUE ==========
Only in df1 : 0
Only in df3 : 7388


In [4]:
import numpy as np

def is_empty_mid_labels(x):
    # real Python list / numpy array
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x) == 0

    # normal missing values
    if x is None:
        return True

    if isinstance(x, float) and pd.isna(x):
        return True

    # string versions
    if isinstance(x, str):
        return x.strip() in ["", "[]", "nan", "None", "null"]

    return False


def get_one_column_error_rows(data):
    mid_empty = data["mid_labels"].apply(is_empty_mid_labels)
    mid_exists = ~mid_empty

    boss_str = data["bosserhof_class"].astype(str).str.strip()

    boss_empty = (
        data["bosserhof_class"].isna()
        | boss_str.isin(["", "nan", "None", "null"])
    )

    boss_exists = ~boss_empty

    return data[
        (mid_empty & boss_exists) |
        (mid_exists & boss_empty)
    ].copy()

In [5]:
df1_one_col_errors = get_one_column_error_rows(df1)
df3_one_col_errors = get_one_column_error_rows(df3)

s1 = set(df1["gml_id"])
s3 = set(df3["gml_id"])

only_df3_rows = df3[df3["gml_id"].isin(s3 - s1)].copy()

rerun_df = pd.concat(
    [df1_one_col_errors, df3_one_col_errors, only_df3_rows],
    ignore_index=True
).drop_duplicates(subset="gml_id", keep="first")

print("df1 one-column errors:", len(df1_one_col_errors))
print("df3 one-column errors:", len(df3_one_col_errors))
print("only in df3:", len(only_df3_rows))
print("final rerun rows:", len(rerun_df))

rerun_df.head()

df1 one-column errors: 5452
df3 one-column errors: 5452
only in df3: 7388
final rerun rows: 12840


,index,gml_id,volume_m3,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,...,information,tags_search,additional_information,website,sentence,interpreted_type,mid_labels,bosserhof_class,mid_label,geometry
0,12.0,124,67.272774,Buildings for business or commerce,None,residential,None,Gebäude,residence;industrial,None,...,None,None,None,None,general_building_context: building_label=Build...,business or commerce building,[],normal office,None,MULTIPOLYGON Z (((609326.321 5796857.112 73.95...
1,13.0,126,216.046139,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,general_building_context: building_label=Build...,business or commerce building,[],normal office,None,MULTIPOLYGON Z (((609865.109 5797289.743 81.44...
2,14.0,210,826.397281,Buildings for business or commerce,None,residential,None,Gebäude,None,None,...,None,None,None,None,general_building_context: building_label=Build...,business or commerce building,[],normal office,None,MULTIPOLYGON Z (((609350.767 5796511.791 78.12...
3,15.0,211,120.328826,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,general_building_context: building_label=Build...,business or commerce building,[],normal office,None,MULTIPOLYGON Z (((609653.832 5797216.449 76.53...
4,37.0,493,160.383023,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,...,None,None,None,None,general_building_context: building_label=Build...,commercial building,[],Services,None,MULTIPOLYGON Z (((609135.888 5797128.686 80.26...


In [6]:
import json
import numpy as np
import pandas as pd

rerun_clean = rerun_df.drop(
    columns=["mid_label", "geometry"],
    errors="ignore"
).copy()

def convert_mid_labels(x):

    # missing values
    if x is None:
        return "[]"

    if isinstance(x, float) and pd.isna(x):
        return "[]"

    # numpy array -> list
    if isinstance(x, np.ndarray):
        x = x.tolist()

    # tuple -> list
    if isinstance(x, tuple):
        x = list(x)

    # list -> JSON string
    if isinstance(x, list):
        return json.dumps(x)

    # already string
    return str(x)

rerun_clean["mid_labels"] = rerun_clean["mid_labels"].apply(convert_mid_labels)

rerun_clean.to_parquet(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\error_predictions-3.parquet",
    index=False
)

print("Saved successfully")

Saved successfully


In [7]:
df = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun.gpkg"
)

df1 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-0.gpkg"
)

df2 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-1.gpkg"
)

df3 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-2.gpkg"
)

df4 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-3.gpkg"
)

df5 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-4.gpkg"
)

df6 = gpd.read_file(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-5.gpkg"
)


print(len(df))
print(len(df1))
print(len(df2))
print(len(df3))
print(len(df4))
print(len(df5))
print(len(df6))

561338
27042
13566
2476
700
9891
2228


In [8]:
common_gml_ids = (
    set(df["gml_id"])
    & set(df2["gml_id"])
    & set(df3["gml_id"])
    & set(df4["gml_id"])
)

print("Common gml_id count in df, df2, df3, df4:", len(common_gml_ids))

# Merge/combine all four dataframes row-wise
merged_df = pd.concat([df, df2, df3, df4], ignore_index=True)

# Remove duplicate gml_id rows if needed
merged_df = merged_df.drop_duplicates(subset="gml_id", keep="first")

print("Merged dataframe rows:", len(merged_df))

Common gml_id count in df, df2, df3, df4: 0
Merged dataframe rows: 578080


In [9]:
def replace_rows_by_gml_id(merged_df, replacement_df, name="replacement_df"):
    merged_updated = merged_df.copy()
    replacement = replacement_df.copy()

    merged_updated["gml_id"] = merged_updated["gml_id"].astype(str)
    replacement["gml_id"] = replacement["gml_id"].astype(str)

    matched_gml_ids = set(merged_updated["gml_id"]) & set(replacement["gml_id"])

    print(f"Matched gml_id count with {name}:", len(matched_gml_ids))

    common_cols = merged_updated.columns.intersection(replacement.columns)

    merged_updated = merged_updated.set_index("gml_id")
    replacement = replacement.set_index("gml_id")

    matching_index = merged_updated.index.intersection(replacement.index)
    cols_to_replace = common_cols.drop("gml_id", errors="ignore")

    merged_updated.loc[matching_index, cols_to_replace] = replacement.loc[
        matching_index,
        cols_to_replace
    ]

    merged_updated = merged_updated.reset_index()

    print(f"Rows replaced from {name}:", len(matching_index))

    return merged_updated

In [10]:
merged_df = replace_rows_by_gml_id(merged_df, df1, name="df1")
merged_df = replace_rows_by_gml_id(merged_df, df5, name="df5")
merged_df = replace_rows_by_gml_id(merged_df, df6, name="df6")

Matched gml_id count with df1: 27042
Rows replaced from df1: 27042
Matched gml_id count with df5: 9891
Rows replaced from df5: 9891
Matched gml_id count with df6: 2228
Rows replaced from df6: 2228


In [11]:
import ast
import pandas as pd

def is_empty_label(x):
    if x is None or pd.isna(x):
        return True

    if isinstance(x, list):
        return len(x) == 0

    if isinstance(x, str):
        x = x.strip()
        if x == "" or x == "[]":
            return True

        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed) == 0
        except Exception:
            pass

    return False


mid_empty = merged_df["mid_labels"].apply(is_empty_label)
bosserhof_empty = merged_df["bosserhof_class"].apply(is_empty_label)

only_mid_label = (~mid_empty) & bosserhof_empty
only_bosserhof_label = mid_empty & (~bosserhof_empty)
both_empty = mid_empty & bosserhof_empty

interpreted_error = (
    merged_df["interpreted_type"]
    .astype(str)
    .str.contains("error", case=False, na=False)
)

print("Only mid_labels:", only_mid_label.sum())
print("Only bosserhof_label:", only_bosserhof_label.sum())
print("Both mid_labels and bosserhof_label empty:", both_empty.sum())
print("Rows with error in interpreted_column:", interpreted_error.sum())

Only mid_labels: 265
Only bosserhof_label: 4167
Both mid_labels and bosserhof_label empty: 312199
Rows with error in interpreted_column: 0


In [12]:
only_one_label_df = merged_df.loc[
    only_mid_label | only_bosserhof_label
].copy()

print("Rows with only mid_labels or only bosserhof_class:", len(only_one_label_df))

lastrerun = pd.read_parquet(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\error_predictions-5.parquet"
)

print(len(lastrerun))


Rows with only mid_labels or only bosserhof_class: 4432
721


In [13]:
# Make sure gml_id types match
only_one_label_df["gml_id"] = only_one_label_df["gml_id"].astype(str)
lastrerun["gml_id"] = lastrerun["gml_id"].astype(str)

# Common gml_id values
common_gml_ids = set(only_one_label_df["gml_id"]) & set(lastrerun["gml_id"])

print("Common gml_id count:", len(common_gml_ids))

# Rows from only_one_label_df that are also in lastrerun
common_only_one_label_df = only_one_label_df[
    only_one_label_df["gml_id"].isin(common_gml_ids)
].copy()

# Rows from lastrerun that are also in only_one_label_df
common_lastrerun_df = lastrerun[
    lastrerun["gml_id"].isin(common_gml_ids)
].copy()

print("Common rows in only_one_label_df:", len(common_only_one_label_df))
print("Common rows in lastrerun:", len(common_lastrerun_df))


Common gml_id count: 721
Common rows in only_one_label_df: 721
Common rows in lastrerun: 721


In [14]:
only_one_label_df.columns

Index(['gml_id', 'index', 'volume_m3', 'label_en', 'osm_building_type',
       'osm_landuse_class', 'osm_landuse_name', 'gfk_class',
       'ALKIS_Landuse_info', 'osm_names', 'alkis_address', 'email', 'amenity',
       'building', 'shop', 'tourism', 'information', 'tags_search',
       'additional_information', 'website', 'sentence', 'interpreted_type',
       'mid_labels', 'bosserhof_class', 'error', 'mid_label', 'geometry',
       'gml_id_pred', 'short_reason'],
      dtype='object')

In [15]:
only_one_label_df = only_one_label_df.drop(columns=['interpreted_type',
       'mid_labels', 'bosserhof_class', 'error', 'mid_label', 'geometry',
       'gml_id_pred', 'short_reason'], errors="ignore").copy()

In [16]:
only_one_label_df.to_parquet(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\only_one_label_rows.parquet",
    index=False)